# ANVAYA symptom-to-condition baseline

This notebook ranks the dataset's recorded conditions from structured symptom evidence. It is an educational experiment on simulated patients and is not clinically validated. Do not use these scores for diagnosis, treatment, or emergency decisions.

Data source: Tchango et al., *DDXPlus: A New Dataset For Automatic Medical Diagnosis* (NeurIPS 2022), CC BY. Download the English train and test ZIP files and `release_evidences.json` from [Figshare](https://figshare.com/articles/dataset/DDXPlus_Dataset_English_/22687585). Keep the files local. The notebook uses the official train/test separation and a reproducible random sample from each split for laptop execution.

Install dependencies in the notebook environment with `%pip install pandas scikit-learn`. Set `DATA_DIR` below to the directory with the release files. The validation split is unused in this baseline. Run all cells to obtain metrics.

Reference execution used pandas 2.2.3 and scikit-learn 1.8.0.

In [1]:
import ast
import json
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

DATA_DIR = Path("ddxplus")
SEED = 42
TRAIN_LIMIT = 50_000
TEST_LIMIT = 10_000


def load_split(split, limit):
    stem = DATA_DIR / f"release_{split}_patients"
    csv_path = stem.with_suffix(".csv")
    path = csv_path if csv_path.is_file() else stem.with_suffix(".zip")
    if not path.is_file():
        raise FileNotFoundError(f"Download and place the {split} release at {path}")
    frame = pd.read_csv(path, usecols=["EVIDENCES", "PATHOLOGY"])
    if frame.isna().any().any():
        raise ValueError(f"Missing evidence or target in {split} split")
    return frame.sample(n=min(limit, len(frame)), random_state=SEED)

## Encode symptom evidence

Keep explicit categorical or multi-choice responses as separate indicator features. Exclude antecedents using the release metadata. Use `PATHOLOGY` solely as the target. Exclude the supplied differential diagnosis, demographic fields, and all test data from model fitting.

In [2]:
with (DATA_DIR / "release_evidences.json").open(encoding="utf-8") as source:
    evidence_metadata = json.load(source)

symptom_codes = {
    code for code, item in evidence_metadata.items() if not item["is_antecedent"]
}


def symptom_features(tokens):
    return {
        token: 1
        for token in tokens
        if token.split("_@_", 1)[0] in symptom_codes
    }


train = load_split("train", TRAIN_LIMIT)
test = load_split("test", TEST_LIMIT)
if set(test["PATHOLOGY"]) - set(train["PATHOLOGY"]):
    raise ValueError("Increase TRAIN_LIMIT to include all sampled test labels.")

vectorizer = DictVectorizer()
x_train = vectorizer.fit_transform(
    symptom_features(ast.literal_eval(value)) for value in train["EVIDENCES"]
)
x_test = vectorizer.transform(
    symptom_features(ast.literal_eval(value)) for value in test["EVIDENCES"]
)
y_train = train["PATHOLOGY"]
y_test = test["PATHOLOGY"]
print(f"Train cases: {len(train):,} | Test cases: {len(test):,}")

Train cases: 50,000 | Test cases: 10,000


## Compare baselines on held-out synthetic cases

Top-3 hit rate measures how often the recorded condition appears among the three highest-scored classes. Ties follow model class order. Model scores are uncalibrated for real patients. The majority baseline provides a reference for accuracy and macro F1.

In [3]:
majority = DummyClassifier(strategy="most_frequent").fit(x_train, y_train)
model = LogisticRegression(max_iter=1000, random_state=SEED).fit(x_train, y_train)

predicted = model.predict(x_test)
scores = model.predict_proba(x_test)
ranked = model.classes_[(-scores).argsort(axis=1, kind="stable")[:, :3]]
top_three = (ranked == y_test.to_numpy()[:, None]).any(axis=1).mean()

for name, prediction in [
    ("Majority", majority.predict(x_test)),
    ("Logistic regression", predicted),
]:
    print(f"{name}: accuracy={accuracy_score(y_test, prediction):.3f}, "
          f"macro F1={f1_score(y_test, prediction, average='macro'):.3f}")
print(f"Logistic regression top-3 hit rate: {top_three:.3f}")
print(classification_report(y_test, predicted, zero_division=0))

Majority: accuracy=0.065, macro F1=0.002
Logistic regression: accuracy=0.985, macro F1=0.979
Logistic regression top-3 hit rate: 1.000
                                          precision    recall  f1-score   support

     Acute COPD exacerbation / infection       1.00      0.67      0.80       160
                Acute dystonic reactions       1.00      1.00      1.00       253
                        Acute laryngitis       0.99      0.97      0.98       230
                      Acute otitis media       1.00      1.00      1.00       272
                   Acute pulmonary edema       1.00      1.00      1.00       206
                    Acute rhinosinusitis       0.93      0.75      0.83       148
                      Allergic sinusitis       0.99      1.00      1.00       172
                             Anaphylaxis       1.00      1.00      1.00       273
                                  Anemia       1.00      1.00      1.00       505
                     Atrial fibrillation    

## Rank conditions for one simulated example

This demo uses symptom codes from a held-out simulated case. For an interactive demo, replace the list with symptom codes from the released evidence dictionary. Free-text symptom parsing is outside this experiment.

In [4]:
def rank_conditions(symptom_tokens):
    features = vectorizer.transform([symptom_features(symptom_tokens)])
    if not features.nnz:
        raise ValueError("Provide at least one symptom token seen during training.")
    scores = model.predict_proba(features)[0]
    indices = (-scores).argsort(kind="stable")[:3]
    return pd.DataFrame({
        "condition": model.classes_[indices],
        "model_score": scores[indices],
    })


for value in test["EVIDENCES"]:
    tokens = ast.literal_eval(value)
    if vectorizer.transform([symptom_features(tokens)]).nnz:
        print("Example symptom codes:", list(symptom_features(tokens)))
        display(rank_conditions(tokens))
        break
else:
    print("No sampled test case contains recognized symptoms.")

Example symptom codes: ['E_66', 'E_201', 'E_214']
                               condition  model_score
Bronchospasm / acute asthma exacerbation     0.784874
     Acute COPD exacerbation / infection     0.213106
                          Bronchiectasis     0.000461


## Limitations

The simulated generator and closed set of conditions limit what held-out scores tell us. Real-world evaluation would require independent, representative clinical data, calibration, and clinician review. The notebook is separate from the FastAPI application.